In [0]:

import boto3
import os

os.environ["AWS_ACCESS_KEY_ID"]     = "your-access-key-id"
os.environ["AWS_SECRET_ACCESS_KEY"] = "your-secret-access-key"
os.environ["AWS_DEFAULT_REGION"]    = "eu-north-1"

spark.conf.set("fs.s3a.access.key", "your-access-key-id")
spark.conf.set("fs.s3a.secret.key", "your-secret-access-key")
spark.conf.set("fs.s3a.endpoint",   "s3.eu-north-1.amazonaws.com")

print("AWS Credentials Configured")

In [0]:
# ============================================================
#   CREATE ALL SCHEMAS
#   Catalog: social_catalog
# ============================================================

spark.sql("CREATE SCHEMA IF NOT EXISTS social_catalog.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS social_catalog.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS social_catalog.gold")

spark.sql("SHOW SCHEMAS IN social_catalog").show()

print("✅ All 3 Schemas Created Successfully")

In [0]:
# ============================================================
#   CREATE BRONZE EMPTY TABLES
#   Catalog: social_catalog.bronze
# ============================================================

# --- sentiment_table ---
spark.sql("""
    CREATE TABLE IF NOT EXISTS social_catalog.bronze.sentiment_table (
        tweet_id         STRING,
        topic_category   STRING,
        tweet_timestamp  STRING,
        sentiment_score  DOUBLE,
        positive_score   DOUBLE,
        negative_score   DOUBLE,
        neutral_score    DOUBLE,
        impressions      DOUBLE,
        likes            DOUBLE,
        engagement_count DOUBLE,
        ingested_at      TIMESTAMP
    )
    USING DELTA
""")
print("✅ sentiment_table created")

# --- trends_table ---
spark.sql("""
    CREATE TABLE IF NOT EXISTS social_catalog.bronze.trends_table (
        trend_timestamp  STRING,
        topic_catagory   STRING,
        country          STRING,
        tweet_volume     DOUBLE,
        mention_count    DOUBLE,
        retweet_count    DOUBLE,
        trend_score      DOUBLE,
        sentiment_index  DOUBLE,
        impressions      DOUBLE,
        engagement_count DOUBLE,
        ingested_at      TIMESTAMP
    )
    USING DELTA
""")
print("✅ trends_table created")

# --- tweets_table ---
spark.sql("""
    CREATE TABLE IF NOT EXISTS social_catalog.bronze.tweets_table (
        tweet_id         STRING,
        user_id          DOUBLE,
        tweet_text       STRING,
        timestamp        STRING,
        timestamp_1      STRING,
        likes            DOUBLE,
        retweets         DOUBLE,
        replies          DOUBLE,
        impressions      DOUBLE,
        engagement       DOUBLE,
        ingested_at      TIMESTAMP
    )
    USING DELTA
""")
print("✅ tweets_table created")

# --- user_metadata_table ---
spark.sql("""
    CREATE TABLE IF NOT EXISTS social_catalog.bronze.user_metadata_table (
        user_id              DOUBLE,
        country              STRING,
        topic_catagory       STRING,
        account_created_date STRING,
        followers_count      DOUBLE,
        following_count      DOUBLE,
        likes_count          DOUBLE,
        shares_count         DOUBLE,
        posts_count          DOUBLE,
        varified             STRING,
        ingested_at          TIMESTAMP
    )
    USING DELTA
""")
print("✅ user_metadata_table created")

# --- valid_table ---
spark.sql("""
    CREATE TABLE IF NOT EXISTS social_catalog.bronze.valid_table (
        tweet_id         STRING,
        topic_category   STRING,
        tweet_text       STRING,
        tweet_timestamp  STRING,
        impressions      DOUBLE,
        likes            DOUBLE,
        retweets         DOUBLE,
        replies          DOUBLE,
        engagement_count DOUBLE,
        sentiment_score  DOUBLE,
        ingested_at      TIMESTAMP
    )
    USING DELTA
""")
print("✅ valid_table created")

spark.sql("SHOW TABLES IN social_catalog.bronze").show()
print("✅ All Bronze Tables Created Successfully")

In [0]:
# ============================================================
#   PUSH DATA: S3 → BRONZE
#   FIXED: Partitioned folder structure + schema defined
#   Path: s3://realtime-parquetfiles/trends/2026/03/13/05/
# ============================================================

from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import *

s3_base = "s3://realtime-parquetfiles"

# ============================================================
# DEFINE SCHEMAS MANUALLY (fixes UNABLE_TO_INFER_SCHEMA error)
# ============================================================

sentiment_schema = StructType([
    StructField("tweet_id",         StringType(),  True),
    StructField("topic_category",   StringType(),  True),
    StructField("tweet_timestamp",  StringType(),  True),
    StructField("sentiment_score",  DoubleType(),  True),
    StructField("positive_score",   DoubleType(),  True),
    StructField("negative_score",   DoubleType(),  True),
    StructField("neutral_score",    DoubleType(),  True),
    StructField("impressions",      DoubleType(),  True),
    StructField("likes",            DoubleType(),  True),
    StructField("engagement_count", DoubleType(),  True)
])

trends_schema = StructType([
    StructField("trend_timestamp",  StringType(),  True),
    StructField("topic_catagory",   StringType(),  True),
    StructField("country",          StringType(),  True),
    StructField("tweet_volume",     DoubleType(),  True),
    StructField("mention_count",    DoubleType(),  True),
    StructField("retweet_count",    DoubleType(),  True),
    StructField("trend_score",      DoubleType(),  True),
    StructField("sentiment_index",  DoubleType(),  True),
    StructField("impressions",      DoubleType(),  True),
    StructField("engagement_count", DoubleType(),  True)
])

tweets_schema = StructType([
    StructField("tweet_id",         StringType(),  True),
    StructField("user_id",          DoubleType(),  True),
    StructField("tweet_text",       StringType(),  True),
    StructField("timestamp",        StringType(),  True),
    StructField("timestamp_1",      StringType(),  True),
    StructField("likes",            DoubleType(),  True),
    StructField("retweets",         DoubleType(),  True),
    StructField("replies",          DoubleType(),  True),
    StructField("impressions",      DoubleType(),  True),
    StructField("engagement",       DoubleType(),  True)
])

user_metadata_schema = StructType([
    StructField("user_id",               DoubleType(),  True),
    StructField("country",               StringType(),  True),
    StructField("topic_catagory",        StringType(),  True),
    StructField("account_created_date",  StringType(),  True),
    StructField("followers_count",       DoubleType(),  True),
    StructField("following_count",       DoubleType(),  True),
    StructField("likes_count",           DoubleType(),  True),
    StructField("shares_count",          DoubleType(),  True),
    StructField("posts_count",           DoubleType(),  True),
    StructField("varified",              StringType(),  True)
])

valid_schema = StructType([
    StructField("tweet_id",         StringType(),  True),
    StructField("topic_category",   StringType(),  True),
    StructField("tweet_text",       StringType(),  True),
    StructField("tweet_timestamp",  StringType(),  True),
    StructField("impressions",      DoubleType(),  True),
    StructField("likes",            DoubleType(),  True),
    StructField("retweets",         DoubleType(),  True),
    StructField("replies",          DoubleType(),  True),
    StructField("engagement_count", DoubleType(),  True),
    StructField("sentiment_score",  DoubleType(),  True)
])


# ============================================================
# READ FROM S3 WITH RECURSIVE LOOKUP
# Handles partitioned structure: folder/year/month/day/hour/
# ============================================================

sentiment_df = spark.read \
    .format("parquet") \
    .option("recursiveFileLookup", "true") \
    .option("mergeSchema", "true") \
    .schema(sentiment_schema) \
    .load(f"{s3_base}/sentiment/")

trends_df = spark.read \
    .format("parquet") \
    .option("recursiveFileLookup", "true") \
    .option("mergeSchema", "true") \
    .schema(trends_schema) \
    .load(f"{s3_base}/trends/")

tweets_df = spark.read \
    .format("parquet") \
    .option("recursiveFileLookup", "true") \
    .option("mergeSchema", "true") \
    .schema(tweets_schema) \
    .load(f"{s3_base}/tweets/")

user_metadata_df = spark.read \
    .format("parquet") \
    .option("recursiveFileLookup", "true") \
    .option("mergeSchema", "true") \
    .schema(user_metadata_schema) \
    .load(f"{s3_base}/user_metadata/")

valid_df = spark.read \
    .format("parquet") \
    .option("recursiveFileLookup", "true") \
    .option("mergeSchema", "true") \
    .schema(valid_schema) \
    .load(f"{s3_base}/valid/")


# ============================================================
# VERIFY S3 SOURCE COUNTS
# ============================================================

print("===== S3 SOURCE ROW COUNTS =====")
print(f"sentiment/     : {sentiment_df.count()}")
print(f"trends/        : {trends_df.count()}")
print(f"tweets/        : {tweets_df.count()}")
print(f"user_metadata/ : {user_metadata_df.count()}")
print(f"valid/         : {valid_df.count()}")


# ============================================================
# WRITE TO BRONZE DELTA TABLES
# ============================================================

ingested_at = current_timestamp()

sentiment_df \
    .withColumn("ingested_at", ingested_at) \
    .write.format("delta").mode("append") \
    .saveAsTable("social_catalog.bronze.sentiment_table")
print("✅ sentiment_table loaded")

trends_df \
    .withColumn("ingested_at", ingested_at) \
    .write.format("delta").mode("append") \
    .saveAsTable("social_catalog.bronze.trends_table")
print("✅ trends_table loaded")

tweets_df \
    .withColumnRenamed("timestamp_1", "timestamp.1") \
    .withColumn("ingested_at", ingested_at) \
    .write.format("delta").mode("append") \
    .saveAsTable("social_catalog.bronze.tweets_table")
print("✅ tweets_table loaded")

user_metadata_df \
    .withColumn("ingested_at", ingested_at) \
    .write.format("delta").mode("append") \
    .saveAsTable("social_catalog.bronze.user_metadata_table")
print("✅ user_metadata_table loaded")

valid_df \
    .withColumn("ingested_at", ingested_at) \
    .write.format("delta").mode("append") \
    .saveAsTable("social_catalog.bronze.valid_table")
print("✅ valid_table loaded")


# ============================================================
# VERIFY BRONZE COUNTS AFTER WRITE
# ============================================================

print("\n===== BRONZE ROW COUNTS AFTER INGESTION =====")
print(f"sentiment_table     : {spark.table('social_catalog.bronze.sentiment_table').count()}")
print(f"trends_table        : {spark.table('social_catalog.bronze.trends_table').count()}")
print(f"tweets_table        : {spark.table('social_catalog.bronze.tweets_table').count()}")
print(f"user_metadata_table : {spark.table('social_catalog.bronze.user_metadata_table').count()}")
print(f"valid_table         : {spark.table('social_catalog.bronze.valid_table').count()}")

print("\n✅ S3 → Bronze Ingestion Complete")

In [0]:
%sql
-- Paste each query directly in Databricks SQL Editor

SELECT COUNT(*) FROM social_catalog.bronze.sentiment_table;
SELECT COUNT(*) FROM social_catalog.bronze.trends_table;
SELECT COUNT(*) FROM social_catalog.bronze.tweets_table;
SELECT COUNT(*) FROM social_catalog.bronze.user_metadata_table;
SELECT COUNT(*) FROM social_catalog.bronze.valid_table;

-- Preview data
SELECT * FROM social_catalog.bronze.sentiment_table     LIMIT 5;
SELECT * FROM social_catalog.bronze.trends_table        LIMIT 5;
SELECT * FROM social_catalog.bronze.tweets_table        LIMIT 5;
SELECT * FROM social_catalog.bronze.user_metadata_table LIMIT 5;
SELECT * FROM social_catalog.bronze.valid_table         LIMIT 5;